# Frozen Feature kNN Model Comparison

Run one or more frozen DINO-style backbones on the same dataset fraction. Smoke runs are not saved; full-dataset runs can cache features and save compact result tables.

In [1]:
import os

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: c:\Users\vasko\Downloads\xai\project


In [2]:
import gc

import pandas as pd
import torch
import transformers
from torch.utils.data import DataLoader

from src.data import get_small_data
from src.model import extract_feature_bank, get_dino_backbone, model_metadata
from src.results import (
    feature_bank_exists,
    load_feature_bank,
    make_run_dir,
    save_feature_bank,
    save_knn_outputs,
    validate_save_request,
)
from src.train import (
    DEFAULT_KNN_FEWSHOT_SETTINGS,
    filter_knn_fewshot_settings,
    run_knn_fewshot_experiment,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

## Config

In [3]:
dataset_name = "chestmnist"
data_fraction = 0.03
dataset_seed = 0
save_outputs = False
# Partial saved runs go under outputs/<dataset>/_partial/...
# Full saved runs go under outputs/<dataset>/<run_name>/.

threshold = 0.05
knn_batch_size = 256
seeds = list(range(10))
output_root = project_root / "outputs"

model_configs = [
    {
        "run_name": "dinov2_small_224",
        "model_name": "facebook/dinov2-small",
        "image_size": 224,
        "batch_size": 32,
        "enabled": True,
    },
    {
        "run_name": "dinov2_large_224",
        "model_name": "facebook/dinov2-large",
        "image_size": 224,
        "batch_size": 16,
        "enabled": True,
    },
    {
        "run_name": "stanford_dinov2_xray_224",
        "model_name": "StanfordAIMI/dinov2-base-xray-224",
        "image_size": 224,
        "batch_size": 16,
        "enabled": True,
    },
    {
        "run_name": "rad_dino_518",
        "model_name": "microsoft/rad-dino",
        "image_size": 518,
        "batch_size": 4,
        "enabled": True,
    },
]

validate_save_request(save_outputs, data_fraction)
models_to_run = [config for config in model_configs if config.get("enabled", True)]
models_to_run

[{'run_name': 'dinov2_small_224',
  'model_name': 'facebook/dinov2-small',
  'image_size': 224,
  'batch_size': 32,
  'enabled': True},
 {'run_name': 'dinov2_large_224',
  'model_name': 'facebook/dinov2-large',
  'image_size': 224,
  'batch_size': 16,
  'enabled': True},
 {'run_name': 'stanford_dinov2_xray_224',
  'model_name': 'StanfordAIMI/dinov2-base-xray-224',
  'image_size': 224,
  'batch_size': 16,
  'enabled': True},
 {'run_name': 'rad_dino_518',
  'model_name': 'microsoft/rad-dino',
  'image_size': 518,
  'batch_size': 4,
  'enabled': True}]

## Run Experiments

In [ ]:
all_summaries = []
all_per_class_full = []
all_metadata = []

for model_config in models_to_run:
    run_name = model_config["run_name"]
    model_name = model_config["model_name"]
    image_size = model_config["image_size"]
    batch_size = model_config["batch_size"]
    run_dir = make_run_dir(
        output_root,
        dataset_name,
        run_name,
        data_fraction=data_fraction,
        dataset_seed=dataset_seed,
    )

    print("\n" + "=" * 80)
    print(run_name)
    print(model_name)

    train_loader, val_loader, class_names = get_small_data(
        batch_size=batch_size,
        image_size=image_size,
        data_fraction=data_fraction,
        seed=dataset_seed,
    )
    train_feature_loader = DataLoader(
        train_loader.dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )
    val_feature_loader = DataLoader(
        val_loader.dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    context = {
        "dataset_name": dataset_name,
        "run_name": run_name,
        "model_name": model_name,
        "image_size": image_size,
    }

    if save_outputs and feature_bank_exists(run_dir):
        print("loading cached features")
        feature_bank, feature_metadata = load_feature_bank(run_dir)
    else:
        model = get_dino_backbone(model_name).to(device)
        backbone_metadata = model_metadata(model)
        print("total params:", f"{backbone_metadata['total_params']:,}")
        print("trainable params:", f"{backbone_metadata['trainable_params']:,}")

        feature_bank, feature_metadata = extract_feature_bank(
            model,
            train_feature_loader,
            val_feature_loader,
            device,
            class_names=class_names,
        )
        feature_metadata = {
            **context,
            **backbone_metadata,
            **feature_metadata,
        }

        del model
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

        if save_outputs:
            save_feature_bank(run_dir, feature_bank, metadata=feature_metadata)

    settings = filter_knn_fewshot_settings(
        DEFAULT_KNN_FEWSHOT_SETTINGS,
        feature_bank["train_features"].shape[0],
    )
    knn_outputs = run_knn_fewshot_experiment(
        train_features=feature_bank["train_features"],
        train_labels=feature_bank["train_labels"],
        val_features=feature_bank["val_features"],
        val_labels=feature_bank["val_labels"],
        class_names=feature_bank["class_names"],
        settings=settings,
        seeds=seeds,
        threshold=threshold,
        batch_size=knn_batch_size,
        device=device,
        context=context,
    )

    metadata = {
        **context,
        "data_fraction": data_fraction,
        "dataset_seed": dataset_seed,
        "batch_size": batch_size,
        "knn_batch_size": knn_batch_size,
        "threshold": threshold,
        "n_train": int(feature_bank["train_features"].shape[0]),
        "n_val": int(feature_bank["val_features"].shape[0]),
        "device": device,
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
        **feature_metadata,
        **knn_outputs["metadata"],
    }

    if save_outputs:
        save_knn_outputs(
            run_dir,
            runs_df=knn_outputs["runs"],
            summary_df=knn_outputs["summary"],
            per_class_full_df=knn_outputs["per_class_full"],
            metadata=metadata,
        )

    all_summaries.append(knn_outputs["summary"])
    all_per_class_full.append(knn_outputs["per_class_full"])
    all_metadata.append(metadata)

combined_summary_df = pd.concat(all_summaries, ignore_index=True)
combined_per_class_full_df = pd.concat(all_per_class_full, ignore_index=True)
combined_summary_df


dinov2_small_224
facebook/dinov2-small
Using downloaded and verified file: C:\Users\vasko\.medmnist\chestmnist.npz
Using downloaded and verified file: C:\Users\vasko\.medmnist\chestmnist.npz
total params: 22,056,576
trainable params: 0


Extracting features:   0%|          | 0/74 [00:00<?, ?it/s]

c:\Users\vasko\anaconda3\envs\cumid\Lib\site-packages\transformers\models\dinov2\modeling_dinov2.py:258: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  context_layer = torch.nn.functional.scaled_dot_product_attention(


Extracting features:   0%|          | 0/11 [00:00<?, ?it/s]

## Compare Full Reference Rows

In [ ]:
full_rows_df = combined_summary_df[combined_summary_df["setting"] == "full"].copy()
full_rows_df[
    [
        "run_name",
        "model_name",
        "image_size",
        "n_train",
        "k",
        "mean_auc_mean",
        "f1_macro_mean",
        "f1_micro_mean",
        "mean_accuracy_mean",
        "exact_match_accuracy_mean",
    ]
]

In [ ]:
metadata_df = pd.DataFrame(all_metadata)
metadata_df[
    [
        "run_name",
        "model_name",
        "image_size",
        "total_params",
        "trainable_params",
        "feature_dim",
        "feature_extraction_seconds",
        "knn_eval_seconds",
        "peak_gpu_memory_mb",
    ]
]

## Visualize Model Comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot_summary_df = combined_summary_df.copy()
plot_summary_df["display_name"] = plot_summary_df["run_name"].str.replace("_", " ")

plot_metadata_df = metadata_df.copy()
plot_metadata_df["display_name"] = plot_metadata_df["run_name"].str.replace("_", " ")

### Few-Shot Curves by Model

In [ ]:
metrics_to_plot = [
    ("mean_auc_mean", "mean_auc_std", "Mean AUROC"),
    ("f1_macro_mean", "f1_macro_std", "Macro F1"),
    ("f1_micro_mean", "f1_micro_std", "Micro F1"),
    ("mean_accuracy_mean", None, "Mean Accuracy"),
    ("exact_match_accuracy_mean", None, "Exact Match Accuracy"),
]

fewshot_plot_df = plot_summary_df[plot_summary_df["setting"] != "full"].copy()

fig, axes = plt.subplots(2, 3, figsize=(17, 8))
axes = axes.ravel()

for ax, (mean_col, std_col, title) in zip(axes, metrics_to_plot):
    for display_name, group_df in fewshot_plot_df.groupby("display_name", sort=False):
        group_df = group_df.sort_values("n_train")
        x = group_df["n_train"].astype(float).to_numpy()
        y = group_df[mean_col].astype(float).to_numpy()

        ax.plot(x, y, marker="o", linewidth=2, label=display_name)

        if std_col is not None and group_df[std_col].notna().any():
            std = group_df[std_col].astype(float).fillna(0).to_numpy()
            runs = group_df["runs"].astype(float).to_numpy()
            ci95 = 1.96 * std / np.sqrt(runs)
            ax.fill_between(x, y - ci95, y + ci95, alpha=0.12)

    ax.set_xscale("log")
    ax.set_xlabel("Number of training/reference images")
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

axes[-1].axis("off")
handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="lower center", ncol=2)
fig.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()

### Full-Reference Metrics

In [ ]:
full_plot_df = plot_summary_df[plot_summary_df["setting"] == "full"].copy()
full_plot_df = full_plot_df.sort_values("mean_auc_mean", ascending=False)

full_metrics = [
    ("mean_auc_mean", "Mean AUROC"),
    ("f1_macro_mean", "Macro F1"),
    ("f1_micro_mean", "Micro F1"),
    ("mean_accuracy_mean", "Mean Accuracy"),
]

fig, axes = plt.subplots(1, len(full_metrics), figsize=(18, 4), sharey=True)

for ax, (metric_col, title) in zip(axes, full_metrics):
    ax.barh(full_plot_df["display_name"], full_plot_df[metric_col])
    ax.invert_yaxis()
    ax.set_title(title)
    ax.set_xlabel(title)
    ax.grid(True, axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

### Efficiency vs Performance

In [ ]:
efficiency_df = full_plot_df.merge(
    plot_metadata_df,
    on=["dataset_name", "run_name", "model_name", "image_size", "display_name"],
    how="left",
    suffixes=("", "_metadata"),
)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

scatter_specs = [
    ("total_params", "Total Parameters", True),
    ("feature_extraction_seconds", "Feature Extraction Seconds", False),
    ("peak_gpu_memory_mb", "Peak GPU Memory MB", False),
]

for ax, (x_col, x_label, log_x) in zip(axes, scatter_specs):
    plot_df = efficiency_df.dropna(subset=[x_col, "mean_auc_mean"])
    ax.scatter(plot_df[x_col], plot_df["mean_auc_mean"], s=80)

    for _, row in plot_df.iterrows():
        ax.text(row[x_col], row["mean_auc_mean"], row["display_name"], fontsize=8, alpha=0.85)

    if log_x:
        ax.set_xscale("log")
    ax.set_xlabel(x_label)
    ax.set_ylabel("Full-Reference Mean AUROC")
    ax.set_title(f"AUROC vs {x_label}")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

efficiency_df[
    [
        "run_name",
        "mean_auc_mean",
        "f1_macro_mean",
        "total_params",
        "trainable_params",
        "feature_dim",
        "feature_extraction_seconds",
        "knn_eval_seconds",
        "peak_gpu_memory_mb",
    ]
]

### Per-Class Full-Reference Diagnostics

In [ ]:
selected_run_name = full_plot_df.iloc[0]["run_name"]

per_class_full_df = combined_per_class_full_df[
    combined_per_class_full_df["run_name"] == selected_run_name
].copy()

print("selected_run_name:", selected_run_name)
per_class_full_df

In [ ]:
if not per_class_full_df.empty:
    plot_df = per_class_full_df.sort_values("auc", ascending=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

    axes[0].barh(plot_df["class_name"], plot_df["auc"])
    axes[0].set_title(f"Per-Class AUROC: {selected_run_name}")
    axes[0].set_xlabel("AUROC")
    axes[0].grid(True, axis="x", alpha=0.3)

    axes[1].barh(plot_df["class_name"], plot_df["f1"])
    axes[1].set_title(f"Per-Class F1: {selected_run_name}")
    axes[1].set_xlabel("F1")
    axes[1].grid(True, axis="x", alpha=0.3)

    plt.tight_layout()
    plt.show()